In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
SOURCE_TABLE = "fuel_project_dev.bronze.fuel_station"
TARGET_TABLE = "fuel_project_dev.silver.fuel_station"

In [0]:
spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
            fuel_station_id STRING,
            latitude DOUBLE,
            longitude DOUBLE,
            manager_name STRING,
            fuel_support ARRAY<STRING>,
            employee_count INT,
            area_sqft INT,
            _silver_ingestion_ts TIMESTAMP
        )
        USING DELTA
        CLUSTER BY (fuel_station_id)
        """)

In [0]:
bronze_df = spark.table(SOURCE_TABLE)

In [0]:
display(bronze_df)

In [0]:
parsed_df = (
    bronze_df
        .withColumn(
            "latitude",
            regexp_extract("location", r"\(([^,]+),", 1).cast("double")
        )
        .withColumn(
            "longitude",
            regexp_extract("location", r",\s*([^)]+)\)", 1).cast("double")
        )
)


In [0]:
parsed_df = (
    parsed_df
        .withColumn(
            "fuel_support",
            split(
                regexp_replace("FuelSupport", r"[\[\]' ]", ""),
                ","
            )
        )
)


In [0]:
silver_ready_df = (
    parsed_df
        .withColumnRenamed("FuelStationID", "fuel_station_id")
        .withColumnRenamed("ManagerName", "manager_name")
        .withColumn("employee_count", col("employee_count").cast("int"))
        .withColumn("area_sqft", col("AreaSqft").cast("double"))
        .withColumn("_silver_ingestion_ts", current_timestamp())
        .select(
            "fuel_station_id",
            "latitude",
            "longitude",
            "manager_name",
            "fuel_support",
            "employee_count",
            "area_sqft",
            "_silver_ingestion_ts"
        )
)


In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(
    spark, TARGET_TABLE
)

(
    silver_table.alias("t")
    .merge(
        silver_ready_df.alias("s"),
        "t.fuel_station_id = s.fuel_station_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)


In [0]:
%sql
SELECT * FROM fuel_project_dev.silver.fuel_station